In [1]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import pytorch_lightning as pl
import yaml 
from pathlib import Path

os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

torch.set_float32_matmul_precision('medium')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

In [2]:
360/5

72.0

In [3]:
config_path = Path("config/binaural_attn/word_task_v10_main_feature_gain_config.yaml")
ckpt_path = Path("attn_cue_models/word_task_v10_main_feature_gain_config/checkpoints/epoch=1-step=24679-v1.ckpt")


config = yaml.load(open(config_path, 'r'), Loader=yaml.FullLoader)

config['num_workers'] = 2
config['hparas']['batch_size'] = 16
config['corpus']['root'] = '/orcd/data/jhm/001/datasets/dataset_binaural_attn/v10/'
config['corpus']['task'] = 'word_and_location'
config['corpus']['return_azim_loc_only'] = True
config['corpus']['feature_gain'] = True
config['corpus']['clean_percentage'] = 1.0

config['model']['num_classes']['num_locs'] = 72

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pytorch_lightning as pl
from typing import Dict
import re
from typing import Optional, Dict

class StageFeatureExtractor(nn.Module):
    """
    Wraps a model and extracts features at a chosen internal stage using a forward hook.
    """
    def __init__(self, backbone: nn.Module, target_stage: str):
        super().__init__()
        self.backbone = backbone
        self.target_stage = target_stage
        self._features = None

        # Register hook at target stage
        for name, module in self.backbone.named_modules():
            if target_stage in name:
                module.register_forward_hook(self._hook)
                break
        else:
            raise ValueError(f"Stage {target_stage} not found in backbone")

    def _hook(self, module, input, output):
        self._features = output

    def forward(self, **inputs):
        _ = self.backbone(**inputs)  # pass everything to backbone
        return self._features



class MultiClassifierModule(pl.LightningModule):
    def __init__(self, 
                 backbone: nn.Module,
                 audio_transforms: nn.Module,
                 target_stage: str,
                 task_dict: Dict[str, int],
                 lr: float = 1e-3):
        super().__init__()
        self.save_hyperparameters(ignore=["backbone"])
        self.task_dict = task_dict
        self.task_names = list(task_dict.keys())
        self.audio_transforms = audio_transforms
        self.automatic_optimization = False  # <--- required for multi-optimizer manual training

        # Feature extractor
        self.feature_extractor = StageFeatureExtractor(backbone, target_stage)
        for p in self.feature_extractor.parameters():
            p.requires_grad = False

        # Infer feature dim
        with torch.no_grad():
            dummy = torch.randn(1, 2, 110_250, device='cuda')
            feats = self.feature_extractor(mixture=dummy)
            feat_dim = feats.view(1, -1).shape[1]

        # Classifier heads
        self.classifier_heads = nn.ModuleDict({
            task: nn.Linear(feat_dim, num_classes)
            for task, num_classes in task_dict.items()
        })

    def forward(self, **inputs):
        feats = self.feature_extractor(**inputs)
        return feats.view(feats.size(0), -1)

    def training_step(self, batch, batch_idx):
        cue_features, cue_mask_ixs, loc_task_ixs, scene_features, labels = batch
        targets = {task_name:labels[:,i] for i, task_name in enumerate(self.task_names)}
        feats = self(mixture=scene_features)

        for i, task_name in enumerate(self.task_names):
            optimizer = self.optimizers()[i]
            optimizer.zero_grad()

            head = self.classifier_heads[task_name]
            logits = head(feats)
            loss = F.cross_entropy(logits, targets[task_name])

            self.manual_backward(loss)
            optimizer.step()

            self.log(f"train_loss_{task_name}", loss, prog_bar=True)

    def validation_step(self, batch, batch_idx):
        cue_features, cue_mask_ixs, loc_task_ixs, scene_features, labels = batch
        targets = {task_name:labels[:,i] for i, task_name in enumerate(self.task_names)}
        feats = self(mixture=scene_features)
        
        total_loss = 0
        for task_name in self.task_names:
            logits = self.classifier_heads[task_name](feats)
            loss = F.cross_entropy(logits, targets[task_name])
            self.log(f"val_loss_{task_name}", loss, prog_bar=True)
            total_loss += loss

        self.log("val_loss_total", total_loss, prog_bar=True)
        return total_loss

    def configure_optimizers(self):
        return [
            torch.optim.Adam(self.classifier_heads[task].parameters(), lr=self.hparams.lr)
            for task in self.task_names
        ]

    def predict_step(self, batch, batch_idx):
        x = batch if isinstance(batch, torch.Tensor) else batch[0]
        feats = self(mixture=x)
        return {
            task: torch.softmax(self.classifier_heads[task](feats), dim=1)
            for task in self.task_names
        }



def list_stage_names(model: nn.Module):
    """
    Print all available module names inside a model so you can pick a target stage.
    """
    print("Available stages in model:\n")
    for name, module in model.named_modules():
        print(f"{name:30s} ({module.__class__.__name__})")

def list_act_names(model: nn.Module):
    """
    Print all available module names inside a model so you can pick a target stage.
    """
    # print("Available stages in model:\n")
    name_list = []
    for name, module in model.named_modules():
        if 'ReLU' in module.__class__.__name__:
            # print(f"{name:30s} ({module.__class__.__name__})")
            # get only the section with conv_block_X 
            if 'conv' in name:
                name = re.search(r'conv_block_[0-9]+', name).group(0)
            elif 'relufc' in name:
                name = 'relufc'
            name_list.append(name)
    return name_list


def format_stage_name(name: str):
    """
    Convert a stage name to a valid Python attribute name.
    """
    if 'conv' in name:
        name = f"_orig_mod.model_dict.{name}.2" # 2 is the ReLU layer after conv
    elif 'fc' in name:
        name = "_orig_mod.relufc"
    return name

class ModelWithFrontEnd(nn.Module):
    def __init__(self,front_end, model):
        super().__init__()
        self.front_end = front_end
        self.model = model

    def forward(self, cue: torch.Tensor = None, 
                mixture: torch.Tensor = None,
                cue_mask_ixs: torch.tensor = None): 
        cue, mixture = self.front_end(cue, mixture)
        return self.model(cue, mixture, cue_mask_ixs)


In [5]:
from src.spatial_attn_lightning import BinauralAttentionModule 

# 1. Load pretrained model and split it
# module = BinauralAttentionModule.load_from_checkpoint(config=config, checkpoint_path=ckpt_path, strict=False)

# Step 1: Create model architecture
module = BinauralAttentionModule(config=config).cuda()

# Step 2: Load the checkpoint manually
checkpoint = torch.load(ckpt_path, map_location='cuda')  # or 'cuda' if needed

# Step 3: Load the weights only (ignore compilation and trainer states)
module.load_state_dict(checkpoint["state_dict"], strict=False)


cochgram = module.coch_gram
cnn = module.model
audio_transforms = module.audio_transforms 
backbone = ModelWithFrontEnd(cochgram, cnn)

# 2. Choose a layer index for feature extraction
act_names = list_act_names(backbone)
layer_to_get = act_names[5]  # for example

# 3. Remove classifier layer to prevent interference
backbone.model.classification = nn.Identity()

batch = next(iter(module.train_dataloader()))
# # 4. Create full Lightning model
model = MultiClassifierModule(
    backbone=backbone,
    target_stage=layer_to_get,
    audio_transforms=audio_transforms,
    task_dict={
        "num_azim_classes": 512,
        "num_word_classes": 800,
    },
    lr=1e-3
)

# 5. Train
trainer = pl.Trainer(accelerator="gpu",
                     devices=1,
                     max_epochs=10)
trainer.fit(model, train_dataloaders=module.train_dataloader(), val_dataloaders=module.val_dataloader())


/orcd/data/jhm/001/om2/imgriff/projects/torch_2_aud_attn/phaselocknet_torch/peripheral_model.py:421: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd(cast_inputs=torch.float32)


Using explicit dim specification for demeaning in audio transforms
Batch in dataloader = False
Using BinauralAuditoryAttentionCNN
v08 True
num_classes={'num_words': 800, 'num_locs': 72}
Model performing both location and word tasks
Using singe gain function per layer
Conv block order: LN -> Conv -> ReLU
fc_attn: True
coch_affine: True
Using dataset BinauralAttentionDataset
BinauralAuditoryAttentionCNN(
  (model_dict): ModuleDict(
    (norm_coch_rep): LayerNorm((2, 40, 20000), eps=1e-05, elementwise_affine=True)
    (attn0): SimpleAttentionalGain()
    (conv_block_0): Sequential(
      (0): LayerNorm((2, 40, 20000), eps=1e-05, elementwise_affine=True)
      (1): Conv2d(2, 32, kernel_size=(2, 34), stride=(1, 1), bias=False)
      (2): ReLU()
    )
    (hann_pool_0): HannPooling2d()
    (attn1): SimpleAttentionalGain()
    (conv_block_1): Sequential(
      (0): LayerNorm((32, 20, 4992), eps=1e-05, elementwise_affine=True)
      (1): Conv2d(32, 64, kernel_size=(2, 14), stride=(1, 1), bias=

/orcd/data/jhm/001/om2/imgriff/projects/torch_2_aud_attn/src/audio_transforms.py:338: UserWarning: "kaiser_window" resampling method name is being deprecated and replaced by "sinc_interp_kaiser" in the next release. The default behavior remains unchanged.
  self.downsampling_op = self.downsampling(self.sr,
/tmp/ipykernel_1882381/4073878853.py:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_s

Using v06 dataset
/orcd/data/jhm/001/datasets/dataset_binaural_attn/v10/
Using 0.1 cue free data
Using gender balanced training 4M set
cue type: mixed
mixture_percentages={'voice_only': 0.5, 'voice_and_location': 0.5}
735 files in train concat dataset
len training set = 229320


/orcd/data/jhm/001/imgriff/conda_envs/pytorch_2/lib/python3.12/site-packages/pytorch_lightning/utilities/parsing.py:209: Attribute 'audio_transforms' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['audio_transforms'])`.
/orcd/data/jhm/001/imgriff/conda_envs/pytorch_2/lib/python3.12/site-packages/lightning_fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /orcd/data/jhm/001/imgriff/conda_envs/pytorch_2/lib/ ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/orcd/data/jhm/001/imgriff/conda_envs/pytorch_2/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has bee

...

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name              | Type                  | Params | Mode 
--------------------------------------------------------------------
0 | audio_transforms  | AudioCompose          | 0      | train
1 | feature_extractor | StageFeatureExtractor | 62.7 M | train
2 | classifier_heads  | ModuleDict            | 320 M  | train
--------------------------------------------------------------------
320 M     Trainable params
62.7 M    Non-trainable params
383 M     Total params
1,532.331 Total estimated model params size (MB)
66        Modules in train mode
0         Modules in eval mode


len training set = 229320
Using v06 dataset
/orcd/data/jhm/001/datasets/dataset_binaural_attn/v10/
cue type: mixed
mixture_percentages={'voice_only': 0.5, 'voice_and_location': 0.5}
49 files in val concat dataset


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/orcd/data/jhm/001/imgriff/conda_envs/pytorch_2/lib/python3.12/site-packages/pytorch_lightning/utilities/data.py:79: Trying to infer the `batch_size` from an ambiguous collection. The batch size we found is 16. To avoid any miscalculations, use `self.log(..., batch_size=batch_size)`.
/opt/conda/conda-bld/pytorch_1729647378361/work/aten/src/ATen/native/cuda/Loss.cu:250: nll_loss_forward_reduce_cuda_kernel_2d: block: [0,0,0], thread: [4,0,0] Assertion `t >= 0 && t < n_classes` failed.
/opt/conda/conda-bld/pytorch_1729647378361/work/aten/src/ATen/native/cuda/Loss.cu:250: nll_loss_forward_reduce_cuda_kernel_2d: block: [0,0,0], thread: [5,0,0] Assertion `t >= 0 && t < n_classes` failed.
/opt/conda/conda-bld/pytorch_1729647378361/work/aten/src/ATen/native/cuda/Loss.cu:250: nll_loss_forward_reduce_cuda_kernel_2d: block: [0,0,0], thread: [11,0,0] Assertion `t >= 0 && t < n_classes` failed.
/opt/conda/conda-bld/pytorch_1729647378361/work/aten/src/ATen/native/cuda/Loss.cu:250: nll_loss_forward_r

RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
